# Notebook 25 — Expanded accuracy and entry-signal decision research

**Objective:** Run a more substantial, predeclared model comparison and produce an explicit **do-not-deploy / research-only** decision report. This is not a search for a winner on the historical holdout.

**Context from Notebook 24:** Accuracy policy averaged **41.05%** vs majority baseline **41.96%**; entry precision **41.43%**, coverage **7.13%**, estimated net return **−0.18%**. No evidence yet of a deployable signal.

**Rules:** qcut(7) maps lowest-to-highest septiles to `[2,2,0,0,1,1,1]`; class 1 = Up. Balanced training is the default, with explicitly named unweighted controls. Three outer and two inner chronological folds. Inner-only policy selection; no holdout access. No prospective ledger writes.

**Caveat:** A next-bar return averaged across all stock-signals is **not** an executable portfolio backtest. Class 1 may include nonpositive returns. Do not trade these signals on this notebook alone.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Run from the repository root or notebooks directory')
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.decision_research import CANDIDATES_25,run_decision_research,summarize_policies,candidate_stability
print('Repository:',ROOT)


## 1. Predeclared experiment

18 configurations across logistic regression, Random Forest, Extra Trees, and histogram gradient boosting; varied regularization, tree leaf sizes, balanced vs unweighted controls, and feature families. All candidates are fixed **before** the outer results are observed. A nested evaluation chooses the accuracy policy and entry policy independently on inner folds.


In [ ]:
N_OUTER=3
N_INNER=2
MAX_TRAIN_ROWS=60000
TREE_TRAIN_ROWS=40000
MIN_COVERAGE=.05
MIN_ENTRIES=200
COST_BPS=10
OUTPUT=ROOT/'artifacts'/'decision_research'/'1d'
print('Candidate models:',len(CANDIDATES_25))
print('Inner model fits:',len(CANDIDATES_25)*N_OUTER*N_INNER)
display(pd.DataFrame([c.__dict__ for c in CANDIDATES_25]))


## 2. Load frozen historical master table

This notebook does **not** ingest fresh data or read the historical holdout for evaluation. `run_decision_research` creates the development split internally.


In [ ]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
found=next((p for p in paths if all((p/f).exists() for f in required)),None)
if found is None: raise FileNotFoundError('Master-table artifacts not found. Run Notebook 03 first.')
master,latest,bundle,manifest=load_master(found)
assert isinstance(master,pd.DataFrame),type(master)
print('Master rows:',len(master),'Master directory:',found)


## 3. Run full nested experiment

**Long-running**: 108 inner model fits plus up to six outer policy fits. Logs show each outer-fold selection. Do not change candidates or thresholds after seeing outer results.


In [ ]:
inner,outer,choices,decision,stability,cutoff=run_decision_research(
    master,output_dir=OUTPUT,n_outer=N_OUTER,n_inner=N_INNER,
    max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,
    min_coverage=MIN_COVERAGE,min_entries=MIN_ENTRIES,cost_bps=COST_BPS)
print('Development cutoff:',cutoff)
print('Inner rows:',len(inner),'Outer policy rows:',len(outer))
display(choices)


## 4. Primary decision dashboard (outer folds)

Evaluate policy-level metrics, not cherry-picked individual outer-fold models. The entry gate is a **conservative diagnostic**, not proof of profitable trading. Failure to pass means **do not promote**.


In [ ]:
display(outer[['outer_fold','policy','candidate','accuracy','majority_baseline_accuracy','balanced_accuracy','roc_auc','up_precision','up_recall','coverage','precision_up','precision_lift','mean_entry_net_return']])
display(decision)
if decision.loc[decision.policy=='entry','entry_gate_pass'].fillna(False).any():
    print('Entry gate met on limited development folds; requires prospective paper validation before deployment.')
else:
    print('NO ENTRY POLICY PROMOTION: research-only. No live-trading evidence.')


## 5. Which model families and features appear promising? (inner diagnostics only)

The table below is **not an unbiased test estimate**. Use it to plan a future predeclared experiment, not to retroactively change Notebook 25's selection.


In [ ]:
display(stability)
family=pd.DataFrame([c.__dict__ for c in CANDIDATES_25])
display(stability.merge(family,left_on='candidate',right_on='name').groupby(['family','feature_set','balanced'],as_index=False).agg(mean_accuracy=('mean_accuracy','mean'),mean_auc=('mean_auc','mean'),mean_fit_seconds=('mean_fit_seconds','mean')).sort_values('mean_accuracy',ascending=False))


## 6. Entry threshold coverage diagnostics (inner folds only)

Very high apparent precision at very low coverage can reflect a handful of selected samples. Inspect entries, baseline Up rate, and coverage together.


In [ ]:
thresholds=(inner.groupby(['candidate','threshold'],as_index=False)
  .agg(n=('n','sum'),entries=('entries','sum'),wins=('up_wins','sum'),base_wins=('up_base_wins','sum')))
thresholds['coverage']=thresholds.entries/thresholds.n
thresholds['up_precision']=thresholds.wins/thresholds.entries.replace(0,float('nan'))
thresholds['baseline_up']=thresholds.base_wins/thresholds.n
thresholds['precision_lift']=thresholds.up_precision-thresholds.baseline_up
display(thresholds.loc[(thresholds.coverage>=MIN_COVERAGE)&(thresholds.entries>=MIN_ENTRIES)].sort_values('precision_lift',ascending=False).head(25))


## 7. Research decision and next investment

If neither policy clears its baseline consistently, stop tuning thresholds and revisit **target economics and feature information**. Candidate directions: sector- and regime-relative factors; market breadth; event-aware volatility; and point-in-time news embeddings. Any news data must be sourced and timestamped in `stock_analytics_v0` before model development.

**Not included:** any new news ingestion, a reopened historical holdout, an executable trading simulator, or retraining the frozen prospective model. Reports are saved under `artifacts/decision_research/1d/`.
